# 5. Pipelines and Automation

sklearn **Pipelines** chain preprocessing steps and models into a single object. This prevents data leakage and makes deployment easier. This notebook covers:
- Basic Pipeline for numerical features
- ColumnTransformer for mixed types
- Full pipeline with model training and evaluation
- Custom transformers
- Cross-validation with pipelines

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

## 5.1 Load the Adult Census Dataset

In [ ]:
url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "adult/adult.data")
cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race",
        "sex", "capital_gain", "capital_loss", "hours_per_week",
        "native_country", "income"]
df = pd.read_csv(url, header=None, names=cols, na_values=" ?",
                 skipinitialspace=True)
y = (df['income'] == '>50K').astype(int)

num_features = ['age', 'education_num', 'capital_gain',
                'capital_loss', 'hours_per_week']
cat_features = ['workclass', 'marital_status', 'occupation',
                'relationship', 'race', 'sex']
print(f"Shape: {df.shape}")

## 5.2 Basic Numerical Pipeline

A Pipeline chains transformers sequentially. Each step receives the output of the previous one.

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

X_num = num_pipe.fit_transform(df[num_features])
print(f"Transformed shape: {X_num.shape}")
print(f"Means (should be ~0): {X_num.mean(axis=0).round(4)}")
print(f"Stds  (should be ~1): {X_num.std(axis=0).round(4)}")

## 5.3 ColumnTransformer for Mixed Types

Real data has both numerical and categorical columns. **ColumnTransformer** applies different pipelines to different column subsets, then concatenates the results.

In [ ]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
])

X = preprocessor.fit_transform(df)
print(f"Input shape:  {df[num_features + cat_features].shape}")
print(f"Output shape: {X.shape}")

## 5.4 Full Pipeline with Model

The final step in a pipeline can be an estimator (classifier or regressor). This ensures the same preprocessing is applied at training and prediction time.

In [ ]:
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    df[num_features + cat_features], y,
    test_size=0.2, random_state=42, stratify=y)

full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['<=50K', '>50K'])}")

## 5.5 Cross-Validation with Pipelines

When you use `cross_val_score` with a pipeline, preprocessing is re-fitted on each training fold. This **prevents data leakage** from the test fold.

In [ ]:
scores = cross_val_score(full_pipeline,
                         df[num_features + cat_features], y,
                         cv=5, scoring='accuracy')
print(f"CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")
print(f"Per-fold: {scores.round(4)}")

## 5.6 Custom Transformers

You can create custom transformers by inheriting from `BaseEstimator` and `TransformerMixin`. They must implement `fit()` and `transform()`.

In [ ]:
class OutlierClipper(BaseEstimator, TransformerMixin):
    """Clip outliers at specified percentiles."""
    def __init__(self, lower_pct=1, upper_pct=99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        X_arr = np.array(X)
        self.lower_ = np.nanpercentile(X_arr, self.lower_pct, axis=0)
        self.upper_ = np.nanpercentile(X_arr, self.upper_pct, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.array(X, dtype=float), self.lower_, self.upper_)

# Test
test_data = np.array([[1, 100], [2, 200], [3, 300], [100, 400], [5, 10000]])
clipper = OutlierClipper(lower_pct=5, upper_pct=95)
clipped = clipper.fit_transform(test_data)
print(f"Before max: {test_data.max(axis=0)}")
print(f"After max:  {clipped.max(axis=0)}")

## 5.7 Pipeline Inspection

After fitting, you can inspect any step of the pipeline to examine learned parameters.

In [ ]:
print("Pipeline steps:")
for name, step in full_pipeline.named_steps.items():
    print(f"  {name}: {type(step).__name__}")

scaler = (full_pipeline.named_steps['preprocessor']
          .named_transformers_['num']
          .named_steps['scaler'])
print(f"\nScaler means: {scaler.mean_.round(2)}")
print(f"Scaler stds:  {scaler.scale_.round(2)}")

## Key Takeaways

1. **Pipelines** chain preprocessing + model into one reproducible object
2. **ColumnTransformer** handles mixed-type data (numerical + categorical)
3. Pipelines in **cross-validation** prevent data leakage
4. **Custom transformers** extend pipelines with domain-specific logic
5. Use `joblib.dump()` / `joblib.load()` to save and deploy pipelines